<a href="https://colab.research.google.com/github/jhhlim/LLMFundamentals/blob/main/Copy_of_Class_Lab_1a_HF_Introduction_to_LLMs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1>Class 1a - Introduction to Large Language Models</h1>

In [1]:
print("Hello Class")

Hello Class


In [ ]:
for k in range(10):
  print(k)

0
1
2
3
4
5
6
7
8
9


# This is text call

---

💡 **NOTE**: We will want to use a GPU to run the examples in this notebook. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**.

---

In [ ]:
%%capture
!pip install transformers>=4.40.1 accelerate>=0.27.2

# Phi-3

The first step is to load our model onto the GPU for faster inference. Note that we load the model and tokenizer separately (although that isn't always necessary).

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=False,
)
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

Although we can now use the model and tokenizer directly, it's much easier to wrap it in a `pipeline` object:

In [ ]:
from transformers import pipeline

# Create a pipeline
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=500,
    do_sample=False
)

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


For more info on "pipeline" class:

https://huggingface.co/docs/transformers/en/main_classes/pipelines

return_full_text=True (Default for text-generation): The pipeline returns the original input text concatenated with the newly generated text.

return_full_text=False: The pipeline returns only the new, generated text, excluding the input prompt.

The max_new_tokens parameter in the Hugging Face transformers pipeline is used to control the maximum number of new tokens generated by a model, ignoring the number of tokens in the input prompt.

do_sample=True: Enables sampling, where the model stochastically selects the next token from the probability distribution over the vocabulary. This introduces randomness, leading to more varied, creative, and diverse outputs. This is often used with other parameters like temperature, top_k, and top_p to control the level of randomness.

do_sample=False: Activates greedy decoding, where the model consistently picks the token with the highest probability at each step. This results in deterministic, predictable, and conservative text generation.

Finally, we create our prompt as a user and give it to the model:

In [ ]:
# The prompt (user input / query)
messages = [
    {"role": "user", "content": "Create a funny joke about chickens."}
]

# Generate output
output = generator(messages)
print(output[0]["generated_text"])

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Why did the chicken join the band? Because it had the drumsticks!


What is the meaning of various roles - user, assistant, system?

https://medium.com/@mudassar.hakim/mastering-prompt-engineering-a-guide-to-system-user-and-assistant-roles-in-openai-api-28fe5fbf1d81


# GPU Savings? From the Menu:
## In the top menu bar, click on Runtime.
## From the drop-down menu, select Disconnect and delete runtime.